In [1]:
%cd ../../
%load_ext dotenv
%dotenv

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [2]:
from pathlib import Path

import polars as pl
import pandas as pd

# Extract POS data to intermediate formats

Columns:
- date: Date
- meal: String
- meal_type: String
- meal_code: String
- restaurant: String
- pcs: Int64
- co2: Float64
- src: String

In [3]:
PATH_DIR_INTER = Path("data/inter/intermediate")

In [4]:
pat_meal_code = r"(\d{1,})"
path_prefix_num = r"\d*\s{1}"

### Extract Kumpula POS file to intermediate format

In [5]:
paths = [
    "data/raw/2025/Sold lunches Kumpula 9-2025.csv",
]

list_dfs = []
for path in paths:
    df = pl.read_csv(path, separator=';')
    df.columns = ['date', 'time', 'restaurant', 'meal_type', 'meal', 'pcs', 'co2']
    list_dfs.append(df)
pos_raw = pl.concat(list_dfs)

pos_raw.head()

date,time,restaurant,meal_type,meal,pcs,co2
str,str,str,str,str,i64,str
"""1.9.2025""","""10:01""","""610 Physicum""","""Liha""","""1510 Panini, Kinkku-salami""",1,"""0,91"""
"""1.9.2025""","""10:28""","""600 Chemicum""","""Vegaani""","""6129 Kasvisjauhispyörykät,toma…",1,"""0,52"""
"""1.9.2025""","""10:32""","""600 Chemicum""","""Liha""","""1317 Lihapullat &Kermaista Dij…",1,"""0,72"""
"""1.9.2025""","""10:33""","""600 Chemicum""","""Liha""","""1317 Lihapullat &Kermaista Dij…",1,"""0,72"""
"""1.9.2025""","""10:34""","""600 Chemicum""","""Liha""","""1317 Lihapullat &Kermaista Dij…",1,"""0,72"""


In [6]:
pos_intermediate_kumpula = (
    pos_raw
    .select(
        pl.col('date').str.to_date(),
        pl.col('time').str.to_time(),
        pl.col('meal').str.replace(path_prefix_num, ""),
        'meal_type',
        pl.col('meal').str.extract(pat_meal_code).alias('meal_code').cast(pl.Int64()),
        'restaurant',
        'pcs',
        pl.col('co2').str.replace(',', '.').str.to_decimal().cast(pl.Float64)
    )

    .group_by('date', 'meal', 'meal_type', 'meal_code', 'restaurant', 'co2')
    .agg(pl.col('pcs').sum())

    .with_columns(
        pl.lit("Sold lunches Kumpula 9-2025.csv").alias('src')
    )
)

pos_intermediate_kumpula.head()

date,meal,meal_type,meal_code,restaurant,co2,pcs,src
date,str,str,i64,str,f64,i64,str
2025-09-19,"""Buffet, henkilöstö""","""Not Mapped""",1899,"""600 Chemicum""",1.26,52,"""Sold lunches Kumpula 9-2025.cs…"
2025-09-23,"""Buffet, henkilöstö""","""Not Mapped""",1899,"""600 Chemicum""",1.89,57,"""Sold lunches Kumpula 9-2025.cs…"
2025-09-18,"""Kreikkalainen salaatti""","""Kasvis""",1001,"""610 Physicum""",1.14,1,"""Sold lunches Kumpula 9-2025.cs…"
2025-09-05,"""Broilernug ja Red curry-mangka""","""Kana""",6881,"""620 Exactum""",4.3,10,"""Sold lunches Kumpula 9-2025.cs…"
2025-09-18,"""Meksikolaista broileria""","""Kana""",1296,"""620 Exactum""",7.32,6,"""Sold lunches Kumpula 9-2025.cs…"


In [7]:
path = PATH_DIR_INTER / "20251027_kumpula.xlsx"
path.parent.mkdir(exist_ok=True, parents=True)

pos_intermediate_kumpula.write_excel(path)

### Extract Viikuna POS file to intermediate format

In [ ]:
paths = [
    # "data/raw/2025/Sold lunches Kumpula 9-2025.csv",
    "data/raw/2025/Data Viikuna 9-2025.csv"
]

list_dfs = []
for path in paths:
    df = pl.read_csv(path, separator=';')
    df.columns = ['pcs', 'date', 'meal', 'waste', 'restaurant']
    list_dfs.append(df)
pos_raw = pl.concat(list_dfs)

pos_raw.head()

pcs,date,meal,waste,restaurant
i64,str,str,str,str
1,"""1.9.2025""","""10093 Vegaani, Take away""","""24,2""","""570 Viikuna"""
1,"""1.9.2025""","""200006 Bar Myöhä Bbq-seitanbow…","""24,2""","""570 Viikuna"""
1,"""1.9.2025""","""3215 Take away ruoka""","""24,2""","""570 Viikuna"""
1,"""1.9.2025""","""9043 Kasvisjalapenonugetteja j…","""24,2""","""570 Viikuna"""
2,"""1.9.2025""","""1009 Pizza, Kasvis""","""24,2""","""570 Viikuna"""


In [9]:
pos_intermediate_vik = (
    pos_raw
    .select(
        pl.col('date').str.to_date(),
        pl.col('meal').str.replace(path_prefix_num, ""),
        pl.lit(None).alias('meal_type').cast(pl.String),
        pl.col('meal').str.extract(pat_meal_code).alias('meal_code').cast(pl.Int64()),
        'restaurant',
        pl.lit(None).alias('co2').cast(pl.Float64),
        'pcs',
        pl.lit("Data Viikuna 9-2025.csv").alias("src")
    )
)

pos_intermediate_vik.head()

date,meal,meal_type,meal_code,restaurant,co2,pcs,src
date,str,str,i64,str,f64,i64,str
2025-09-01,"""Vegaani, Take away""",null,10093,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Bar Myöhä Bbq-seitanbowl""",null,200006,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Take away ruoka""",null,3215,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Kasvisjalapenonugetteja ja tom…",null,9043,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Pizza, Kasvis""",null,1009,"""570 Viikuna""",null,2,"""Data Viikuna 9-2025.csv"""


Save

In [10]:
path = PATH_DIR_INTER / "20251027_viikuna.xlsx"
path.parent.mkdir(exist_ok=True, parents=True)

pos_intermediate_vik.write_excel(path)